<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/09-user-defined-functions/04-avoiding-udfs.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 9.4 — The best UDF is no UDF

The rewrite gallery, live: UDFs from real codebases next to their
native replacements — timed, plan-diffed, and one killer demo of a
UDF destroying parquet predicate pushdown.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("9.4-avoiding-udfs")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = (
    spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
    .withColumn("revenue", F.round(col("quantity") * col("unit_price"), 2))
)
orders.select("order_id", "product", "quantity", "country").show(3)

## Gallery 1: string cleanup (the most common UDF in the wild)

Same output, no Python worker, nulls propagate for free — and
Catalyst can see it.

In [ ]:
import re, time

@F.udf("string")
def clean_udf(s):
    return re.sub(r"[^A-Z0-9]", "", s.strip().upper()) if s else None

clean_native = F.regexp_replace(F.upper(F.trim("product")), r"[^A-Z0-9]", "")

# equivalence first (null-safe compare, 3.5's <=>), then speed
diffs = (orders
         .withColumn("a", clean_udf("product"))
         .withColumn("b", clean_native)
         .where(~col("a").eqNullSafe(col("b")))
         .count())
print("rows that differ:", diffs)

In [ ]:
def bench(label, df):
    start = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label:>16}: {time.perf_counter() - start:5.2f} s")

big = spark.range(2_000_000).withColumn(
    "s", F.concat(F.lit("  code "), (col("id") % 997).cast("string")))

bench("udf clean",    big.select(clean_udf("s")))
bench("native clean", big.select(
    F.regexp_replace(F.upper(F.trim("s")), r"[^A-Z0-9]", "")))

## Gallery 2: date parsing and the if/elif ladder

`datetime.strptime` in a UDF -> to_date with a pattern.
Python if/elif -> when/otherwise (nulls fall to otherwise — decide
that on purpose, 3.5 discipline).

In [ ]:
raw = spark.createDataFrame(
    [("09/07/2026",), ("15/01/2026",), (None,)], "raw_date string")
raw.select("raw_date", F.to_date("raw_date", "dd/MM/yyyy").alias("d")).show()

(orders.select(
    "quantity",
    F.when(col("quantity") >= 10, "bulk")
     .when(col("quantity") >= 5, "multi")
     .otherwise("single").alias("band"))     # null quantity -> "single"! intended?
 .distinct().show())

## Gallery 3: loops over arrays -> higher-order functions

Module 6's transform/filter/aggregate run lambdas INSIDE the JVM.
A `for item in row.items` UDF is never necessary.

In [ ]:
carts = spark.createDataFrame(
    [(1, [(2, 9.99), (1, 24.50)]), (2, [(5, 1.99)]), (3, [])],
    "cart_id int, items array<struct<qty:int, price:double>>")

(carts.withColumn(
    "total",
    F.round(F.aggregate("items", F.lit(0.0),
                        lambda acc, x: acc + x["qty"] * x["price"]), 2))
 .show(truncate=False))
# also in this family: json.loads -> from_json (6.3),
# hashlib -> F.sha2/F.xxhash64, math -> F.pow/F.log/F.round

## The killer demo: a UDF in a filter kills pushdown

Write parquet, filter it two ways, and read the FileScan line.
The native predicate reaches the reader (PushedFilters); the UDF
predicate forces a full scan — 5.2's win, revoked by one wrapper.

In [ ]:
orders.write.mode("overwrite").parquet("output/orders_parquet")
parq = spark.read.parquet("output/orders_parquet")

identity_udf = F.udf(lambda c: c, "string")

parq.where(col("country") == "DE").explain()
# FileScan ... PushedFilters: [IsNotNull(country), EqualTo(country,DE)]

parq.where(identity_udf("country") == "DE").explain()
# FileScan ... PushedFilters: [] -> reads EVERYTHING, then asks Python

In [ ]:
import shutil
shutil.rmtree("output", ignore_errors=True)   # self-clean

## Exercises

1. Hunt the built-in: without a UDF, produce (a) the domain of an
   email column, (b) initials from a full-name column, (c) a
   `days_since_order` column. `spark.sql("SHOW FUNCTIONS")` and the
   F-namespace docs are your catalog.
2. Time the pushdown demo properly: generate a 5M-row parquet table
   partitioned by a category column, then compare the two filters'
   wall time AND the "files read / rows scanned" numbers in the SQL
   tab of the UI.
3. Take `qty_band` from 9.1's notebook (a pandas UDF!) and move it
   to rung 1 of the ladder. What did the rewrite buy that the
   pandas UDF hadn't already bought?
4. Find the rung: (a) `phonenumbers.format_number` per row on 200M
   rows; (b) per-store Prophet forecast; (c) `x.strip().lower()`;
   (d) cumulative revenue per customer (careful — it's not even
   this module).
5. Write the one-paragraph code-review comment you'd leave on a PR
   adding `@F.udf` around `datetime.strptime`, including the exact
   replacement line and two plan-level reasons.

In [ ]:
# your exercise code here